# 31 · Revision-5 evaluation (T4 GPU, resumable; no training)
Read-only for runs and data. Writes `results/p2_rev5/eval/` and the consolidated tables `results/p2_rev5/rev5_*_per_image.csv.gz`.
Run after notebooks 30 A/B (all 200 runs finished), 29 (CSAW-S crops) and 29b (CDD-CESM crops).
* every finished run (seeds 0–4, all four run folders) predicts the frozen cohort, DMID, CSAW-S and CDD-CESM crops;
* new bottleneck runs (seeds 3–4): CBIS-DDSM test fold and DMID with code replacement (as notebook 28, part 3);
* checks: frozen re-evaluation of seeds 0–2 against the stored revision-4 values; CBIS test-fold Dice against each run's own `per_image.csv`.

In [ ]:
# 0 · Settings
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, glob, json, time, gc, hashlib
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
RES      = f'{PROJECT}/results'
OUT      = f'{RES}/p2_rev5'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
SETS     = {'frozen': f'{PROJECT}/data/processed/cbis_mass_256_jit_frozen', 'dmid': f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi',
            'csaws': f'{PROJECT}/data/external/csaws_mass_256_jit_a1', 'cddcesm': f'{PROJECT}/data/external/cddcesm_mass_256_jit'}
ROOTS    = {'train': f'{RES}/p2_jit_runs_train', 'val': f'{RES}/p2_jit_runs_val', 'train4': f'{RES}/p2_rev4_runs_train', 'val4': f'{RES}/p2_rev4_runs_val'}
SCHED    = {'train': 'train', 'val': 'val', 'train4': 'train', 'val4': 'val'}
proto = f'{OUT}/PROTOCOL_rev5_prespecified.txt'
assert os.path.exists(proto), 'protocol missing: run notebook 30 first'
print('protocol', open(proto).read().splitlines()[-1])
missing = [k for k, d in SETS.items() if not os.path.exists(f'{d}/metadata.csv')]
assert not set(missing) & {'frozen', 'dmid'}, f'{missing} missing'
for k in missing: print(f'NOTE: {k} crops not found - run its preparation notebook (29 / 29b) and run this notebook again'); SETS.pop(k)
import glob
QSEG_DIR = f'{PROJECT}/code'                     # qseg package located as in notebooks 13 and 27
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p_ in (QSEG_DIR, f'{PROJECT}/code'):
    if p_ not in sys.path: sys.path.insert(0, p_)
print('qseg package:', QSEG_DIR)
import tensorflow as tf, qseg2, qseg2_rev4; qseg2_rev4.patch()
from qseg.config import Config
cfg = Config(work_dir=OUT); B = cfg.batch_size
print('TF', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))

In [ ]:
# 1 · Data
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
DATA = {}
for k, d in SETS.items():
    m = pd.read_csv(f'{d}/metadata.csv')
    X = np.stack([_png(f'{d}/images/{r}.png', False) for r in m.roi_id]); Y = np.stack([_png(f'{d}/masks/{r}.png', True) for r in m.roi_id])
    extra = {}
    for md_ in sorted(glob.glob(f'{d}/masks_*')):                       # per-reader references (CSAW-S)
        extra[os.path.basename(md_)[6:]] = np.stack([_png(f'{md_}/{r}.png', True) for r in m.roi_id])
    DATA[k] = (X, Y, m, extra); print(k, X.shape, '| patients', m.patient.nunique() if 'patient' in m else '-', '| extra refs', list(extra))
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
def load(d, name):
    tf.keras.backend.clear_session()
    m = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m.load_weights(f'{d}/best.weights.h5'); return m
def runs_of(root):
    return [d for d in sorted(glob.glob(f'{root}/*_fold*_seed*')) if os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5')]
ALL = [(k, d) for k, r in ROOTS.items() for d in runs_of(r)]
print('runs with weights:', len(ALL), '| by seed:', pd.Series([json.load(open(f'{d}/summary.json'))['seed'] for _, d in ALL]).value_counts().sort_index().to_dict())

In [ ]:
# 2 · External sets: every run, every set (resumable)
n = 0
for key, d in ALL:
    todo = [k for k in SETS if not os.path.exists(f'{OUT}/eval/{k}/{key}/{os.path.basename(d)}.csv')]
    if not todo: continue
    s = json.load(open(f'{d}/summary.json')); m = load(d, s['model'])
    for k in todo:
        X, Y, meta, extra = DATA[k]; os.makedirs(f'{OUT}/eval/{k}/{key}', exist_ok=True)
        P = m.predict(X, batch_size=B, verbose=0); t = qseg2.per_image_scores(Y, P, cfg.threshold)
        t.insert(0, 'roi_id', meta.roi_id.values)
        if 'patient' in meta: t.insert(1, 'patient', meta.patient.values)
        t['empty'] = (P[..., 0] >= cfg.threshold).sum((1, 2)) == 0
        for a, Ya in extra.items(): t[f'dice_{a}'] = qseg2.per_image_scores(Ya, P, cfg.threshold).dice.values
        t['set'], t['schedule'], t['model'], t['fold'], t['seed'] = key, SCHED[key], s['model'], int(s['fold']), int(s['seed'])
        t.to_csv(f'{OUT}/eval/{k}/{key}/{os.path.basename(d)}.csv', index=False)
    n += 1; del m; gc.collect()
    if n % 20 == 0: print(time.strftime('%H:%M UTC', time.gmtime()), '| runs evaluated in this session:', n, flush=True)
print('external sets: runs evaluated now', n)

In [ ]:
# 3 · New bottleneck runs (seeds 3-4): CBIS-DDSM test fold and DMID with code replacement (as notebook 28, part 3; resumable)
K = 10
REPL_G = tf.Variable(tf.zeros((B, 4)), trainable=False); REPL_P = tf.Variable(tf.zeros((B, 4, 4, 4)), trainable=False)
USE = tf.Variable(0.0, trainable=False)
if not getattr(qseg2.Sever, '_rev4', False):
    _sever_orig = qseg2.Sever.call
    def _sever_call(self, x):
        y = _sever_orig(self, x); r = REPL_G if len(x.shape) == 2 else REPL_P
        return tf.where(USE > 0.5, tf.cast(r[:tf.shape(x)[0]], y.dtype), y)
    qseg2.Sever.call = _sever_call; qseg2.Sever._rev4 = True
def forward(m):
    f = tf.function(lambda x: m(x, training=False), reduce_retracing=True)
    def run(X, repl=None):
        out = []
        for i in range(0, len(X), B):
            xb = X[i:i + B]
            if repl is None: USE.assign(0.0)
            else:
                r = np.zeros((B,) + repl.shape[1:], np.float32); r[:len(xb)] = repl[i:i + B]
                (REPL_G if repl.ndim == 2 else REPL_P).assign(r); USE.assign(1.0)
            out.append(f(tf.constant(xb)).numpy())
        USE.assign(0.0); return np.concatenate(out)
    return run
def codes_of(m, X):
    cm = tf.keras.Model(m.input, m.get_layer('sever').input)
    return np.concatenate([cm(tf.constant(X[i:i + B]), training=False).numpy() for i in range(0, len(X), B)]).astype(np.float32)
def donor(case_id, r, n):
    return int(np.random.default_rng(int(hashlib.md5(f'{case_id}|{r}'.encode()).hexdigest()[:8], 16)).integers(n))
XD, YD, MD, _ = DATA['dmid']
for key, d in ALL:
    s = json.load(open(f'{d}/summary.json'))
    if int(s['seed']) < 3 or s['model'] in ('full', 'unet'): continue
    dst = f'{OUT}/eval/replace/{key}/{os.path.basename(d)}.csv'
    if os.path.exists(dst): continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    name, fold = s['model'], int(s['fold']); m = load(d, name); run = forward(m)
    tr, _, te = qseg2.fold_split(DFC, fold); tr = tr[np.argsort(DFC.roi_id.values[tr])]
    bank = codes_of(m, XC[tr]); const = bank.mean(0, keepdims=True); parts = []
    for ds, Xs, Ys, ids in (('CBIS', XC[te], YC[te], DFC.roi_id.values[te]), ('DMID', XD, YD, MD.roi_id.values)):
        P0 = run(Xs); t = qseg2.per_image_scores(Ys, P0, cfg.threshold); t.insert(0, 'roi_id', ids)
        t['empty'] = (P0[..., 0] >= cfg.threshold).sum((1, 2)) == 0
        Pc = run(Xs, np.repeat(const, len(Xs), 0)); t['dice_const'] = qseg2.per_image_scores(Ys, Pc, cfg.threshold).dice.values
        t['flip_const'] = ((Pc[..., 0] >= 0.5) != (P0[..., 0] >= 0.5)).mean((1, 2)); t['absdp_const'] = np.abs(Pc - P0)[..., 0].mean((1, 2))
        for r in range(K):
            idx = np.array([donor(c, r, len(bank)) for c in ids]); Ps = run(Xs, bank[idx])
            t[f'dice_swap{r}'] = qseg2.per_image_scores(Ys, Ps, cfg.threshold).dice.values
            if r == 0:
                t['flip_swap'] = ((Ps[..., 0] >= 0.5) != (P0[..., 0] >= 0.5)).mean((1, 2)); t['absdp_swap'] = np.abs(Ps - P0)[..., 0].mean((1, 2))
        t.insert(0, 'dataset', ds); parts.append(t)
    out = pd.concat(parts, ignore_index=True)
    out['set'], out['schedule'], out['model'], out['fold'], out['seed'] = key, SCHED[key], name, fold, int(s['seed'])
    out['epochs_run'], out['best_epoch'], out['dice_training'] = s['epochs_run'], s['best_epoch'], s['dice_per_image']
    out.to_csv(dst, index=False); del m; gc.collect()
    print(key, os.path.basename(d), '| CBIS', round(float(out[out.dataset == 'CBIS'].dice.mean()), 4), 'vs training', round(s['dice_per_image'], 4), flush=True)

In [ ]:
# 4 · Checks: (i) frozen re-evaluation of seeds 0-2 vs revision 4; (ii) CBIS test-fold Dice vs each run's per_image.csv
fz = sorted(glob.glob(f'{OUT}/eval/frozen/*/*.csv'))
new = pd.concat([pd.read_csv(f) for f in fz], ignore_index=True)
old_f = f'{RES}/p2_rev4/rev4_frozen_per_image.csv.gz'
if os.path.exists(old_f):
    old = pd.read_csv(old_f); k = ['set', 'model', 'fold', 'seed', 'roi_id']
    j = new.merge(old[k + ['dice']], on=k, suffixes=('', '_rev4'))
    print('(i) frozen, seeds 0-2: rows compared', len(j), '| max |dDice|', float(np.abs(j.dice - j.dice_rev4).max()))
rp = sorted(glob.glob(f'{OUT}/eval/replace/*/*.csv'))
if rp:
    R = pd.concat([pd.read_csv(f) for f in rp], ignore_index=True); R = R[R.dataset == 'CBIS']; mx = 0.0
    for (key, model, fold, seed), g in R.groupby(['set', 'model', 'fold', 'seed']):
        own = pd.read_csv(f'{ROOTS[key]}/{model}_fold{fold}_seed{seed}/per_image.csv').set_index('roi_id').dice
        mx = max(mx, float(np.abs(g.set_index('roi_id').dice - own.reindex(g.roi_id).values).max()))
    print('(ii) CBIS test fold, new runs: max |dDice| vs per_image.csv', mx)

In [ ]:
# 5 · Consolidate (full precision, gzip)
for k in list(SETS) + ['replace']:
    fs = sorted(glob.glob(f'{OUT}/eval/{k}/*/*.csv'))
    if fs:
        a = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
        a.to_csv(f'{OUT}/rev5_{k}_per_image.csv.gz', index=False, compression='gzip', float_format='%.17g')
        print(k, a.shape, '| runs', a.groupby(['set', 'model', 'fold', 'seed']).ngroups)
# CBIS-DDSM test folds of all runs (each run's own per_image.csv)
rows = []
for key, d in ALL:
    s = json.load(open(f'{d}/summary.json')); t = pd.read_csv(f'{d}/per_image.csv')
    t['set'], t['schedule'], t['model'], t['fold'], t['seed'] = key, SCHED[key], s['model'], int(s['fold']), int(s['seed']); rows.append(t)
c = pd.concat(rows, ignore_index=True); c.to_csv(f'{OUT}/rev5_cbis_per_image.csv.gz', index=False, compression='gzip', float_format='%.17g')
print('cbis', c.shape, '| runs', c.groupby(['set', 'model', 'fold', 'seed']).ngroups)
print(c.groupby(['schedule', 'model', 'seed']).dice.mean().unstack().round(4).to_string())